# 03 · CNN (Fase 3, Prácticas 6, 7 y 8)
ResNet18 con transfer learning y fine-tuning por etapas. Todo en **validación**; el test se usa una sola vez en la Fase 5.

**Un cambio por experimento** (plan): cada corrida usa un YAML de `configs/` y queda registrada en W&B.
- `cnn.yaml`: ResNet18 preentrenada, `pos_weight`. Es el experimento principal.
- `cnn_sampler.yaml`: igual pero con `WeightedRandomSampler` (plan 3.3).
- `cnn_scratch.yaml`: CNN chica sin preentrenar (Práctica 7).

Hipótesis (Práctica 7): el fine-tuning de una red preentrenada supera a la CNN chica desde cero en F2, porque hay solo ~197 defectuosas para aprender.

**Primero `SMOKE = True`** (≈ 2 min) para comprobar que todo corre; luego `False`.

In [ ]:
# --- Bootstrap: same cell at the top of every notebook ---
import os
import subprocess
import sys
from pathlib import Path

REPO = "matifleman/inspectia"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/InspectIA")
    REPO_DIR = Path("/content/inspectia")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git", str(REPO_DIR)], check=True)
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE_ROOT = REPO_DIR
    if "WANDB_API_KEY" not in os.environ:
        os.environ["WANDB_MODE"] = "offline"  # local runs without a key log offline

os.chdir(REPO_DIR)
if IN_COLAB:  # locally, install once in your venv (see README)
    %pip install -q -r requirements.txt -e .
# Editable installs are only picked up after a kernel restart; this makes the
# package importable in the current session too.
sys.path.insert(0, str(REPO_DIR / "src"))

DATA_DIR = DRIVE_ROOT / "data"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
SPLITS_DIR = DRIVE_ROOT / "splits"

from inspectia.utils import get_device, set_seed

SEED = 42
set_seed(SEED)
device = get_device()
print(f"Repo: {REPO_DIR} | Data: {DATA_DIR} | Device: {device}")

In [ ]:
# Paths shared by every notebook after the bootstrap
DATASET_DIR = DATA_DIR / "KolektorSDD2"
SPLITS_DIR = REPO_DIR / "splits"            # versioned in git: the whole team uses the SAME splits
RESULTS_DIR = DRIVE_ROOT / "results"
FEATURES_DIR = DRIVE_ROOT / "features"
for d in (SPLITS_DIR, RESULTS_DIR, FEATURES_DIR, CHECKPOINT_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json, copy
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
from inspectia.data import get_splits, make_loaders, pos_weight
from inspectia.models import build_model, set_stage, count_trainable
from inspectia.train import fit_stage, predict, sigmoid, save_checkpoint
from inspectia.evaluate import (evaluate, best_f2_threshold, bootstrap_ci, trivial_baselines,
                                plot_curves, plot_confusion)
from inspectia.utils import load_config, init_wandb, get_device, set_seed

CONFIG = "cnn.yaml"      # cnn.yaml | cnn_sampler.yaml | cnn_scratch.yaml
SMOKE = True             # True = prueba rápida (1 época, 3 lotes); False = entrenamiento real

cfg = load_config(REPO_DIR / "configs" / CONFIG)
set_seed(cfg["seed"]); device = get_device()
MAX_BATCHES = 3 if SMOKE else None
print("config:", CONFIG, "| device:", device, "| SMOKE:", SMOKE)

splits = get_splits(DATASET_DIR, SPLITS_DIR, cfg["data"]["val_frac"], cfg["seed"])
use_sampler = cfg["data"]["imbalance"] == "sampler"
loaders = make_loaders(DATASET_DIR, splits, cfg["data"]["batch_size"], tuple(cfg["data"]["img_size"]),
                       cfg["data"]["num_workers"], balanced_sampler=use_sampler, seed=cfg["seed"])
PW = None if use_sampler else pos_weight(splits["train"])
print({k: (len(v), int(v["label"].sum())) for k, v in splits.items()}, "| pos_weight:", PW and round(PW, 2))
run = None if SMOKE else init_wandb(cfg, name=cfg["name"], group=cfg["wandb"]["group"], project=cfg["wandb"]["project"])

## 1. Entrenamiento por etapas

In [ ]:
model = build_model(cfg["model"]["arch"], cfg["model"]["pretrained"]).to(device)
history, stage_states = [], {}
for st in cfg["stages"]:
    set_stage(model, st["name"]); print("entrenables: %d de %d" % count_trainable(model))
    history += fit_stage(model, loaders, device, st["name"], 1 if SMOKE else st["epochs"], st["lr"], PW,
                         cfg["train"]["patience"], cfg["train"]["weight_decay"], cfg["train"]["monitor"],
                         MAX_BATCHES, run)
    stage_states[st["name"]] = copy.deepcopy(model.state_dict())

## 2. Curvas de aprendizaje (Práctica 8)
Sobreajuste = la pérdida de train baja y la de validación sube.

In [ ]:
h = pd.DataFrame(history); h["it"] = range(1, len(h) + 1)
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
a.plot(h["it"], h["train_loss"], label="train"); a.plot(h["it"], h["val_loss"], label="val")
a.set(title="Pérdida (BCE)", xlabel="época acumulada"); a.legend(); a.grid(alpha=.3)
b.plot(h["it"], h["val_ap"], label="val AP"); b.plot(h["it"], h["val_f2"], label="val F2 (mejor umbral)")
b.set(title="Validación", xlabel="época acumulada"); b.legend(); b.grid(alpha=.3)
for ax in (a, b):
    for s, g in h.groupby("stage"):
        ax.axvline(g["it"].min() - .5, c="k", alpha=.15)
plt.tight_layout(); h.round(4)

## 3. Evaluación en validación (umbral = el que maximiza F2 en validación)

In [ ]:
y_va, logits = predict(model, loaders["val"], device, MAX_BATCHES)
s_va = sigmoid(logits)
thr = best_f2_threshold(y_va, s_va)
m = evaluate(y_va, s_va, thr); ci = bootstrap_ci(y_va, s_va, thr)
rows = [{"model": cfg["name"], **m, "f2_95CI": tuple(round(v, 2) for v in ci["f2"]), "recall_95CI": tuple(round(v, 2) for v in ci["recall"])}]
base_path = RESULTS_DIR / "baseline_val.json"
if base_path.exists():   # comparación con el baseline de la Fase 2
    bdf = pd.read_json(base_path); bdf = bdf[bdf["model"].str.startswith("resnet18")]; rows.insert(0, bdf.sort_values("avg_precision", ascending=False).iloc[0].to_dict())
display(pd.DataFrame(rows)[["model", "threshold", "f2", "recall", "precision", "auc_roc", "avg_precision", "fn", "fp"]].round(3))
print(pd.DataFrame(trivial_baselines(y_va)).T.round(3))
plot_confusion(m, cfg["name"]);

In [ ]:
# 4) Guardar (solo corrida real): checkpoint, métricas y scores de validación para la Fase 5
if not SMOKE:
    save_checkpoint(model, CHECKPOINT_DIR / f"{cfg['name']}.pt")
    json.dump({"config": cfg, "threshold": thr, "val": m, "val_ci": ci}, open(RESULTS_DIR / f"{cfg['name']}_val.json", "w"), indent=2)
    np.savez(RESULTS_DIR / f"{cfg['name']}_val_scores.npz", y=y_va, scores=s_va)
    h.to_json(RESULTS_DIR / f"{cfg['name']}_history.json", orient="records", indent=2)
    if run:
        run.summary.update({f"val/{k}": v for k, v in m.items() if isinstance(v, float)}); run.finish()
    print("Guardado en", CHECKPOINT_DIR, "y", RESULTS_DIR)
else:
    print("SMOKE: no se guarda nada. Si corrió sin errores, SMOKE = False y repetir desde la celda de configuración.")

## Práctica 7 · registro del experimento controlado
Completar con los números reales de cada corrida (no estimar):

| Corrida (YAML) | Cambio respecto de la anterior | F2 val | Recall val | AP val |
|---|---|---|---|---|
| `cnn_scratch.yaml` | CNN chica, sin preentrenar | | | |
| `cnn.yaml` | ResNet18 preentrenada + fine-tuning | | | |
| `cnn_sampler.yaml` | `WeightedRandomSampler` en vez de `pos_weight` | | | |

Interpretación y próximo paso: *(redactar con los resultados)*.